# 04 SQL Analysis & Database Reporting

**Objective**: Interact with the SQLite database using SQL queries to extract key operational reports and answer clinical questions.

**Inputs**:
- `sql/healthcare_maintenance.db` (populated table: `icu_equipment`)

**Outputs**:
- SQL query results formatted in DataFrames.

In [ ]:
import pandas as pd
from src.utils import run_query

print('Verifying database records...')
count_df = run_query('SELECT COUNT(*) as Total_Records FROM icu_equipment;')
print(count_df)

In [ ]:
# BQ1: Which equipment type fails most frequently?
query_1 = '''
SELECT Equipment_Category, COUNT(*) as Total_Assets, SUM(Device_Failure) as Failure_Count,
       ROUND(CAST(SUM(Device_Failure) AS REAL) / COUNT(*) * 100, 2) as Failure_Rate_Pct
FROM icu_equipment
GROUP BY Equipment_Category
ORDER BY Failure_Count DESC;
'''
df_q1 = run_query(query_1)
print('--- BQ1: Failure Counts ---')
print(df_q1)

In [ ]:
# BQ2: Average telemetry parameters of failed vs healthy machines
query_2 = '''
SELECT Device_Failure, COUNT(*) as Count,
       ROUND(AVG(Ambient_Room_Temp_C), 2) as Avg_Ambient_C,
       ROUND(AVG(Internal_Device_Temp_C), 2) as Avg_Internal_C,
       ROUND(AVG(Cooling_Fan_Speed_RPM), 2) as Avg_Fan_Speed_RPM,
       ROUND(AVG(Motor_Load_Nm), 2) as Avg_Motor_Load_Nm,
       ROUND(AVG(Operating_Hours), 2) as Avg_Operating_Hours,
       ROUND(AVG(Health_Score), 2) as Avg_Health_Score
FROM icu_equipment
GROUP BY Device_Failure;
'''
df_q2 = run_query(query_2)
print('\n--- BQ2: Operational Telemetry ---')
print(df_q2)

In [ ]:
# BQ3: Prioritize High-Risk devices
query_3 = '''
SELECT Product_ID, Equipment_Category, Health_Score, Operating_Hours, Maintenance_Risk
FROM icu_equipment
WHERE Maintenance_Risk = 'High'
ORDER BY Health_Score ASC
LIMIT 10;
'''
df_q3 = run_query(query_3)
print('\n--- BQ3: High Maintenance Risk Priority List ---')
print(df_q3)

In [ ]:
# BQ4: Failure Mode breakdown by Equipment Category
query_4 = '''
SELECT Equipment_Category, COUNT(*) as Total_Devices,
       SUM(Failure_Component_Wear) as Component_Wear_Count,
       SUM(Failure_Overheating) as Overheating_Count,
       SUM(Failure_Power_Supply) as Power_Supply_Count,
       SUM(Failure_Overload) as Overload_Count,
       SUM(Failure_Random_Hardware) as Random_Hardware_Count
FROM icu_equipment
GROUP BY Equipment_Category;
'''
df_q4 = run_query(query_4)
print('\n--- BQ4: Failure Mode Breakdown ---')
print(df_q4)

## Key Findings
- SQL reporting shows clear patterns: Failed equipment averages only a 59.8 Health Score compared to 95.8 for healthy machines.
- A total of 339 failures are distributed among: Component Wear, Overheating, Power Supply, Overload, and Random failures.
- Clinicians can target the 10 lowest health score machines retrieved in BQ3 for immediate proactive maintenance.

## Next Steps
- Build classification models in `06_Model_Building.ipynb` to forecast failure probabilities.